# PA1: 11-hour TinyStories feasibility run

This notebook trains the **19.27M-parameter modern Transformer** on the complete
pretokenized TinyStories stream for **11 hours of training-loop wall time**, then
evaluates validation loss, generates stories, and measures inference latency.

Fixed architecture: vocabulary 8,192; context 256; 4 layers; width 512;
16 query heads; 4 key/value heads; head dimension 32; SwiGLU width 1,344;
RoPE theta 10,000; pre-RMSNorm; untied embeddings.

Before running: enable a **GPU accelerator** and **Internet** in Kaggle. A T4 is
the calibration target. Keep at least 45 minutes of the normal 12-hour session
envelope for setup and post-training evaluation.


## How repository code reaches Kaggle

**Instructor run now (private GitHub repository):** leave `SOURCE_MODE = "git"`.
In Kaggle, open **Add-ons -> Secrets**, create `GITHUB_TOKEN`, grant this notebook
access, and use a read-only fine-grained GitHub token that can read
`alooboii/pa1_advgenai`. Do not paste the token into a cell.

**Student run later (recommended):** create a Kaggle Dataset from the student's
repository snapshot. It may be public or private. Attach that Dataset with
**Add Input**, set `SOURCE_MODE = "kaggle_dataset"`, and set
`KAGGLE_DATASET_SOURCE` to its `/kaggle/input/...` directory. Kaggle inputs are
read-only, so this notebook copies the repository into `/kaggle/working` before
installing or training.

Student repositories can use any internal layout. They only need to replace
`TRAIN_COMMAND`, `CHECKPOINT_PATH`, and `EVALUATE_COMMAND` below with commands
matching the training, checkpoint, and generation interfaces they wrote.


In [ ]:
from pathlib import Path

# "git" for the current private instructor repository; "kaggle_dataset" for
# an attached student repository snapshot.
SOURCE_MODE = "git"
GITHUB_REPOSITORY = "alooboii/pa1_advgenai"
GITHUB_REF = "main"  # Replace with a commit SHA for the final recorded run.
KAGGLE_DATASET_SOURCE = "/kaggle/input/YOUR-DATASET-SLUG"

PROJECT_DIR = Path("/kaggle/working/pa1-modern-transformer")
DATA_REVISION = "3b0e624a62a320d09c1ef9378b7e4f1b6fed6e38"
TRAIN_HOURS = 11
RUN_TESTS = True

# Current instructor repository commands. Students replace these three values
# with commands and paths from their own repositories.
TRAIN_COMMAND = [
    "uv", "run", "python", "train.py",
    "--config", "configs/kaggle_11h.yaml",
    "--set", f"train.max_duration_seconds={TRAIN_HOURS * 60 * 60}",
]
CHECKPOINT_PATH = Path("runs/kaggle_11h/checkpoint_last.pt")
EVALUATE_COMMAND = [
    "uv", "run", "python", "evaluate.py",
    "--config", "configs/kaggle_11h.yaml",
    "--checkpoint", str(CHECKPOINT_PATH),
    "--prompt", "Once upon a time",
    "--max-new-tokens", "256",
]

assert SOURCE_MODE in {"git", "kaggle_dataset"}
assert TRAIN_HOURS == 11
print({"source_mode": SOURCE_MODE, "project": str(PROJECT_DIR), "train_hours": TRAIN_HOURS})


## 1. Runtime preflight


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")
subprocess.run(
    [nvidia_smi, "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    check=True,
)


## 2. Acquire and install the repository


In [ ]:
import base64
from collections import deque

COMMAND_ENV = os.environ.copy()
COMMAND_ENV["UV_CACHE_DIR"] = "/kaggle/working/.uv-cache"
COMMAND_ENV["UV_LINK_MODE"] = "copy"
COMMAND_ENV["PYTHONUNBUFFERED"] = "1"
COMMAND_ENV["GIT_TERMINAL_PROMPT"] = "0"
COMMAND_ENV["MPLBACKEND"] = "Agg"


def run_command(arguments, *, cwd=None, label=None):
    shown = label or " ".join(map(str, arguments))
    print(f"$ {shown}", flush=True)
    completed = subprocess.run(
        list(map(str, arguments)), cwd=cwd, env=COMMAND_ENV, text=True, check=False
    )
    if completed.returncode:
        raise RuntimeError(f"command failed with exit code {completed.returncode}: {shown}")
    return completed


def run_logged(arguments, log_path, *, cwd=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    shown = " ".join(map(str, arguments))
    print(f"$ {shown}", flush=True)
    recent = deque(maxlen=40)
    started = time.monotonic()
    with log_path.open("a", encoding="utf-8") as log_stream:
        process = subprocess.Popen(
            list(map(str, arguments)), cwd=cwd, env=COMMAND_ENV,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            log_stream.write(line)
            log_stream.flush()
            stripped = line.rstrip()
            recent.append(stripped)
            if (
                '"validation_loss":' in stripped
                and '"validation_loss": null' not in stripped
            ) or " passed" in stripped or " failed" in stripped:
                print(stripped, flush=True)
        return_code = process.wait()
    elapsed = time.monotonic() - started
    print(f"Finished in {elapsed / 3600:.2f} hours; log: {log_path}", flush=True)
    if return_code:
        print("\nLast log lines:\n" + "\n".join(recent))
        raise RuntimeError(f"command failed with exit code {return_code}: {shown}")


def git_prefix(token):
    if not token:
        return ["git"]
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {credential}"]


if SOURCE_MODE == "git":
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    git = git_prefix(token)
    repository_url = f"https://github.com/{GITHUB_REPOSITORY}.git"
    if not (PROJECT_DIR / ".git").exists():
        run_command(
            [*git, "clone", repository_url, str(PROJECT_DIR)],
            label=f"git clone https://github.com/{GITHUB_REPOSITORY}.git {PROJECT_DIR}",
        )
    run_command([*git, "fetch", "origin", GITHUB_REF], cwd=PROJECT_DIR,
                label=f"git fetch origin {GITHUB_REF}")
    run_command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=PROJECT_DIR)
else:
    source = Path(KAGGLE_DATASET_SOURCE)
    if not source.exists():
        raise FileNotFoundError(f"Attached Kaggle Dataset not found: {source}")
    candidates = [source] if (source / "pyproject.toml").exists() else [
        path.parent for path in source.rglob("pyproject.toml")
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one repository root, found: {candidates}")
    shutil.copytree(candidates[0], PROJECT_DIR, dirs_exist_ok=True)

for required in ("pyproject.toml", "uv.lock"):
    if not (PROJECT_DIR / required).exists():
        raise FileNotFoundError(f"Repository is missing {required}")

os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)


In [ ]:
if shutil.which("uv") is None:
    run_command([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

run_logged(
    ["uv", "sync", "--frozen"],
    PROJECT_DIR / "artifacts" / "kaggle_uv_sync.log",
    cwd=PROJECT_DIR,
)
run_command(
    [
        "uv", "run", "python", "-c",
        "import torch; print(torch.__version__); print(torch.cuda.get_device_name(0)); assert torch.cuda.is_available()",
    ],
    cwd=PROJECT_DIR,
)


## 3. Run correctness tests


In [ ]:
if RUN_TESTS:
    run_logged(
        ["uv", "run", "pytest", "-q"],
        PROJECT_DIR / "artifacts" / "kaggle_pytest.log",
        cwd=PROJECT_DIR,
    )
else:
    print("Skipped by RUN_TESTS=False")


## 4. Download the pinned full TinyStories token streams


In [ ]:
data_dir = PROJECT_DIR / "data" / "tinystories"
train_bin = data_dir / "data" / "train.bin"
validation_bin = data_dir / "data" / "validation.bin"
metadata_path = data_dir / "metadata.json"
tokenizer_path = data_dir / "tokenizer" / "tokenizer.json"
required_files = (train_bin, validation_bin, metadata_path, tokenizer_path)

if any(not path.exists() for path in required_files):
    run_logged(
        [
            "uv", "run", "hf", "download", "alooboii/pa1-tinystories",
            "--repo-type", "dataset", "--revision", DATA_REVISION,
            "--local-dir", str(data_dir), "--include",
            "metadata.json", "tokenizer/tokenizer.json", "data/*.bin",
        ],
        PROJECT_DIR / "artifacts" / "kaggle_data_download.log",
        cwd=PROJECT_DIR,
    )

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset file is missing after download: {path}")

expected_sizes = {
    train_bin: 933_753_964,
    validation_bin: 9_384_752,
}
for path, expected_size in expected_sizes.items():
    actual_size = path.stat().st_size
    if actual_size != expected_size:
        raise RuntimeError(f"Unexpected size for {path}: {actual_size:,} bytes")
    print(path.relative_to(PROJECT_DIR), f"{actual_size / 2:,.0f} tokens")


## 5. Confirm the 11-hour experiment configuration

The effective batch is `16 x 16 x 256 = 65,536` tokens per optimizer
update. FP16 is used because the calibration GPU is a T4. The optimizer follows
the referenced TinyStories-15M recipe: AdamW, peak learning rate `3e-4`, betas
`(0.9, 0.95)`, weight decay `0.1`, and global gradient clipping at `1.0`.
The first 2% of wall time is warmup; cosine decay reaches `3e-5` at 11 hours.


In [ ]:
config_path = PROJECT_DIR / "configs" / "kaggle_11h.yaml"
if SOURCE_MODE == "git" and not config_path.exists():
    raise FileNotFoundError(
        "configs/kaggle_11h.yaml is missing. Push the current instructor repository changes first."
    )
if config_path.exists():
    print(config_path.read_text())
    run_command(
        [
            "uv", "run", "python", "-c",
            (
                "from modern_transformer.config import load_experiment_config; "
                "from modern_transformer.model import TransformerLM; "
                "c=load_experiment_config('configs/kaggle_11h.yaml'); "
                "m=TransformerLM(c.model); "
                "print('parameters:', f'{m.parameter_count():,}'); "
                "assert m.parameter_count()==19_272_192"
            ),
        ],
        cwd=PROJECT_DIR,
    )


## 6. Train for 11 hours

This is the long-running cell. The instructor trainer stops after 39,600 seconds
of training-loop wall time, performs a final validation pass, and writes a final
checkpoint. Full output goes to `artifacts/kaggle_11h_training.log`; validation
records are also echoed here so convergence remains visible.

Do not start this cell unless enough Kaggle session time remains. Avoid interacting
with the session while it runs.


In [ ]:
expected_finish = time.strftime(
    "%Y-%m-%d %H:%M:%S", time.localtime(time.time() + TRAIN_HOURS * 3600)
)
print("Training begins:", time.strftime("%Y-%m-%d %H:%M:%S"))
print("Expected training-loop finish:", expected_finish)
run_logged(
    TRAIN_COMMAND,
    PROJECT_DIR / "artifacts" / "kaggle_11h_training.log",
    cwd=PROJECT_DIR,
)

import json
summary_path = PROJECT_DIR / CHECKPOINT_PATH.parent / "summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))


## 7. Plot convergence


In [ ]:
import json
import matplotlib.pyplot as plt

metrics_path = PROJECT_DIR / CHECKPOINT_PATH.parent / "metrics.jsonl"
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
train_records = [record for record in records if record.get("train_loss") is not None]
validation_records = [record for record in records if record.get("validation_loss") is not None]

figure, axis = plt.subplots(figsize=(10, 5))
axis.plot(
    [record["tokens"] / 1e6 for record in train_records],
    [record["train_loss"] for record in train_records],
    alpha=0.55, label="training loss",
)
axis.plot(
    [record["tokens"] / 1e6 for record in validation_records],
    [record["validation_loss"] for record in validation_records],
    marker="o", label="validation loss",
)
axis.set(xlabel="Tokens processed (millions)", ylabel="Cross-entropy (nats/token)",
         title="TinyStories convergence: 19.27M-parameter GQA Transformer")
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
convergence_path = PROJECT_DIR / "artifacts" / "kaggle_11h_convergence.png"
figure.savefig(convergence_path, dpi=160)
plt.show()
print("Saved:", convergence_path)


## 8. Validation and one generated story


In [ ]:
checkpoint = PROJECT_DIR / CHECKPOINT_PATH
if not checkpoint.exists():
    raise FileNotFoundError(f"Training checkpoint not found: {checkpoint}")
run_logged(
    EVALUATE_COMMAND,
    PROJECT_DIR / "artifacts" / "kaggle_11h_evaluation.log",
    cwd=PROJECT_DIR,
)
evaluation_path = checkpoint.parent / "evaluation.json"
print(json.dumps(json.loads(evaluation_path.read_text()), indent=2))


## 9. Inference timing and additional generations

The timing below measures the implementation students actually wrote. It reports
batch-one prefill latency at 32, 128, and 256 tokens, plus autoregressive tokens
per second for 128 newly generated tokens. The assignment does not require a KV
cache, so generation recomputes the cropped context at every step.

This cell uses the current instructor module names. Students should replace its
imports and checkpoint loading with their own model interface while keeping the
synchronization and timing procedure.


In [ ]:
import statistics
import sys

sys.path.insert(0, str(PROJECT_DIR / "src"))
import torch
from modern_transformer.checkpoint import load_checkpoint
from modern_transformer.config import load_experiment_config
from modern_transformer.data import load_tokenizer
from modern_transformer.generation import generate
from modern_transformer.model import TransformerLM

config = load_experiment_config(config_path)
device = torch.device("cuda")
model = TransformerLM(config.model).to(device)
load_checkpoint(checkpoint, model=model, map_location=device)
model.eval()
tokenizer = load_tokenizer(data_dir)


def synchronize():
    torch.cuda.synchronize(device)


prefill = {}
with torch.inference_mode():
    for sequence_length in (32, 128, 256):
        tokens = torch.randint(0, 8192, (1, sequence_length), device=device)
        for _ in range(5):
            model(tokens)
        synchronize()
        latencies = []
        for _ in range(20):
            started = time.perf_counter()
            model(tokens)
            synchronize()
            latencies.append((time.perf_counter() - started) * 1000)
        prefill[str(sequence_length)] = {
            "median_ms": statistics.median(latencies),
            "p90_ms": sorted(latencies)[17],
            "tokens_per_second": sequence_length / (statistics.median(latencies) / 1000),
        }

prompts = [
    "Once upon a time",
    "There was a little rabbit named Pip",
    "One day, Mia found a shiny red box",
]
generations = []
torch.cuda.reset_peak_memory_stats(device)
for index, prompt in enumerate(prompts):
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    generator = torch.Generator(device="cuda").manual_seed(42000 + index)
    synchronize()
    started = time.perf_counter()
    output = generate(
        model, input_ids, max_new_tokens=128, temperature=0.8, top_k=50,
        eos_token_id=tokenizer.token_to_id("<|endoftext|>"), generator=generator,
    )
    synchronize()
    elapsed = time.perf_counter() - started
    new_tokens = output.shape[1] - input_ids.shape[1]
    text = tokenizer.decode(output[0].tolist(), skip_special_tokens=False)
    generations.append({
        "prompt": prompt,
        "new_tokens": new_tokens,
        "elapsed_seconds": elapsed,
        "tokens_per_second": new_tokens / elapsed,
        "text": text,
    })
    print(f"\n--- sample {index + 1}: {new_tokens / elapsed:.2f} tokens/s ---\n{text}")

benchmark = {
    "parameter_count": model.parameter_count(),
    "gpu": torch.cuda.get_device_name(device),
    "prefill_batch_1": prefill,
    "autoregressive_generations": generations,
    "peak_cuda_memory_bytes": torch.cuda.max_memory_allocated(device),
}
benchmark_path = PROJECT_DIR / "artifacts" / "kaggle_11h_inference.json"
benchmark_path.write_text(json.dumps(benchmark, indent=2) + "\n")
print("\n", json.dumps({k: v for k, v in benchmark.items() if k != "autoregressive_generations"}, indent=2))


## 10. Preserve the run

Use **Save Version** after the notebook completes so Kaggle preserves files under
`/kaggle/working`. The checkpoint remains in the run directory. The smaller files
below are the core feasibility evidence to download and compare.


In [ ]:
from IPython.display import FileLink, display

evidence = [
    PROJECT_DIR / CHECKPOINT_PATH.parent / "summary.json",
    PROJECT_DIR / CHECKPOINT_PATH.parent / "metrics.jsonl",
    PROJECT_DIR / CHECKPOINT_PATH.parent / "evaluation.json",
    PROJECT_DIR / "artifacts" / "kaggle_11h_convergence.png",
    PROJECT_DIR / "artifacts" / "kaggle_11h_inference.json",
    PROJECT_DIR / "artifacts" / "kaggle_11h_training.log",
]
for path in evidence:
    if path.exists():
        display(FileLink(str(path)))
    else:
        print("Missing:", path)
